In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
model =  ChatOpenAI()

In [25]:
# state
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [26]:
# create node 
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'generate an outline for a blog on the title - {title}'
   
     # ask that qns to llm
    try:
        outline = model.invoke(prompt).content
    except Exception as exc:
        if "insufficient_quota" in str(exc).lower() or "ratelimiterror" in type(exc).__name__.lower():
            outline = "Unable to answer because the OpenAI quota has been exhausted."
        else:
            raise

    # update state
    state['outline'] = outline

    return state

In [27]:
# create node 
def create_blog(state: BlogState) -> BlogState:
    # fetch title
    title = state['title']

    # fetch outline
    outline = state['outline']

    # call llm gen outline
    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'
   
     # ask that qns to llm
    try:
        content = model.invoke(prompt).content
    except Exception as exc:
        if "insufficient_quota" in str(exc).lower() or "ratelimiterror" in type(exc).__name__.lower():
            content = "Unable to answer because the OpenAI quota has been exhausted."
        else:
            raise

    # update state
    state['content'] = content

    return state

In [28]:
graph = StateGraph(BlogState)

# node
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

# compile
workflow  = graph.compile()

In [29]:
# execute
initial_state = {'title': "rise of ai in india"}

try:
    final_state = workflow.invoke(initial_state)
except Exception as exc:
    if "insufficient_quota" in str(exc).lower() or "ratelimiterror" in type(exc).__name__.lower():
        answer = "Unable to answer because the OpenAI quota has been exhausted."
    else:
        raise

print(final_state)
print(final_state['content'])

{'title': 'rise of ai in india', 'outline': 'Unable to answer because the OpenAI quota has been exhausted.', 'content': 'Unable to answer because the OpenAI quota has been exhausted.'}
Unable to answer because the OpenAI quota has been exhausted.
